# Semaine 13 – Évaluation pratique – Partie 2

## Objectifs de la session

À l'issue de cette évaluation pratique (1,5 heure), l'étudiant sera capable de :

- **Comprendre** l'architecture et les fonctions principales de **CMSIS-DSP** sur STM32.
- **Implémenter** des algorithmes de TNS (FFT, filtrage, convolution) dans un contexte **embarqué**.
- **Traiter** des signaux **audio** et **ECG** en temps réel.
- **Optimiser** les performances : mémoire, cycles CPU, précision numérique (Q15, float32).
- **Simuler** en Python les traitements qui seraient exécutés sur STM32.

## Format de l'évaluation

- **Durée** : 1 h 30
- **Modalité** : en laboratoire, sur machine (Python) + cartes STM32 (si disponibles)
- **Documents autorisés** : notes de cours, documentation CMSIS-DSP
- **Barème indicatif** :
  - Partie A – CMSIS-DSP et représentations numériques : **4 pts**
  - Partie B – Filtrage temps réel simulé : **6 pts**
  - Partie C – Traitement audio / ECG : **5 pts**
  - Partie D – Optimisation embarquée : **5 pts**
  - **Total : 20 points**

## Consignes

- Répondez à **toutes** les parties.
- Pour les parties nécessitant du matériel STM32, une **simulation Python** est fournie.
- **Commentez** chaque résultat et comparez avec les implémentations de référence (Python).
- Le corrigé sera fourni en fin de séance.

# 0. Glossaire des acronymes utilisés

| Acronyme | Signification (anglais) | Signification (français) |
|---|---|---|
| **CMSIS** | Cortex Microcontroller Software Interface Standard | Standard d'interface logicielle pour microcontrôleurs Cortex |
| **DSP** | Digital Signal Processing | Traitement Numérique du Signal |
| **MCU** | Microcontroller Unit | Microcontrôleur |
| **HAL** | Hardware Abstraction Layer | Couche d'abstraction matérielle |
| **ISR** | Interrupt Service Routine | Routine de service d'interruption |
| **DMA** | Direct Memory Access | Accès direct à la mémoire |
| **ADC** | Analog-to-Digital Converter | Convertisseur Analogique-Numérique |
| **DAC** | Digital-to-Analog Converter | Convertisseur Numérique-Analogique |
| **FFT** | Fast Fourier Transform | Transformée de Fourier Rapide |
| **FIR** | Finite Impulse Response | Filtre à Réponse Impulsionnelle Finie |
| **IIR** | Infinite Impulse Response | Filtre à Réponse Impulsionnelle Infinie |
| **Q15 / Q31** | Fixed-Point Formats (Q notation) | Formats à virgule fixe |
| **ECG** | Electrocardiogram | Électrocardiogramme |
| **SNR** | Signal-to-Noise Ratio | Rapport Signal / Bruit |
| **MIPS** | Million Instructions Per Second | Millions d'instructions par seconde |
| **Biquad** | Second-Order Section | Section du second ordre (IIR) |

> **Conventions** :
> - $x[n]$ : signal discret ; $h[n]$ : réponse impulsionnelle
> - $y[n]$ : sortie filtrée ; $X[k]$ : DFT de $x[n]$
> - $f_s$ : fréquence d'échantillonnage (Hz)
> - Q15 : entier signé 16 bits avec 15 bits fractionnaires (plage [−1, 1[)

In [ ]:
# Imports communs à toutes les parties
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.fft import fft, ifft, fftfreq
from scipy.signal import find_peaks, freqz, butter, cheby1, filtfilt, lfilter
import time

# Configuration des graphiques
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Reproductibilité
np.random.seed(42)

print("Bibliothèques chargées avec succès.")

# Partie A – CMSIS-DSP et représentations numériques (4 points)

## A.1 – Représentation en virgule fixe Q15

**Question A.1 (2 pts)** : CMSIS-DSP utilise massivement la représentation **Q15** pour les calculs sur MCU (efficacité et faible coût mémoire).

1. Rappelez la plage de valeurs représentables en Q15 (entier signé 16 bits, 1 bit de signe, 15 bits fractionnaires).
2. Implémentez les fonctions `float_to_q15(x)` et `q15_to_float(x_q15)`.
3. Testez avec $x = 0.5$, $-0.75$, $0.999$, $0.001$.
4. Calculez l'erreur de quantification pour chaque valeur.

In [ ]:
# --- Votre code ici ---
def float_to_q15(x):
    """Convertit un flottant dans [-1, 1[ en Q15 (int16)."""
    x = np.clip(x, -1.0, 1.0 - 2**-15)
    return np.int16(np.round(x * 2**15))

def q15_to_float(x_q15):
    """Convertit un Q15 (int16) en flottant."""
    return x_q15.astype(np.float64) / 2**15

# Tests
valeurs = [0.5, -0.75, 0.999, 0.001]
print(f"{'valeur':>10} {'Q15 (int16)':>15} {'retour':>12} {'erreur':>12}")
for v in valeurs:
    q = float_to_q15(v)
    r = q15_to_float(q)
    print(f"{v:>10.5f} {q:>15d} {r:>12.8f} {abs(v-r):>12.2e}")

print(f"\nPlage Q15 : [-1.0, {1 - 2**-15:.8f}]")
print(f"Résolution (pas) : 2^-15 = {2**-15:.2e}")

### Interprétation A.1

- La représentation **Q15** code un flottant de $[-1, 1[$ sur 16 bits signés.
- La **résolution** (pas de quantification) est $\Delta = 2^{-15} \approx 3.05 \times 10^{-5}$.
- L'erreur de quantification est **bornée** par $\Delta/2 \approx 1.53 \times 10^{-5}$.
- **Avantages** : opérations arithmétiques rapides sur MCU sans FPU (ou avec FPU moins coûteux), consommation mémoire réduite (2 octets/valeur).
- **Inconvénients** : risque de **saturation** (débordement) lors d'additions ou multiplications ; il faut donc normaliser les signaux et utiliser des fonctions CMSIS comme `arm_add_q15`, `arm_mult_q15` qui gèrent la saturation.

## A.2 – Fonctions CMSIS-DSP essentielles

**Question A.2 (2 pts)** : Complétez le tableau ci-dessous en indiquant le rôle de chaque fonction CMSIS-DSP et son équivalent Python/NumPy.

| Fonction CMSIS-DSP | Rôle | Équivalent Python |
|---|---|---|
| `arm_fir_init_f32` | ... | ... |
| `arm_fir_f32` | ... | ... |
| `arm_biquad_cascade_df1_f32` | ... | ... |
| `arm_rfft_fast_f32` | ... | ... |
| `arm_cmplx_mag_f32` | ... | ... |
| `arm_max_f32` | ... | ... |
| `arm_dot_prod_f32` | ... | ... |

**Réponse** :

### Réponse A.2

| Fonction CMSIS-DSP | Rôle | Équivalent Python |
|---|---|---|
| `arm_fir_init_f32` | Initialise une instance de filtre FIR (coefficients, buffer d'état) | `signal.lfilter_zi`, initialisation manuelle |
| `arm_fir_f32` | Applique un filtre FIR à un bloc d'échantillons | `signal.lfilter(b, [1.0], x)` |
| `arm_biquad_cascade_df1_f32` | Filtre IIR en cascade de sections biquad (forme directe I) | `signal.lfilter(b, a, x)` |
| `arm_rfft_fast_f32` | Calcule la FFT réelle (plus efficace que la FFT complexe) | `np.fft.rfft(x)` |
| `arm_cmplx_mag_f32` | Calcule le module d'un vecteur complexe | `np.abs(X)` |
| `arm_max_f32` | Trouve le maximum d'un vecteur | `np.max(x)` |
| `arm_dot_prod_f32` | Produit scalaire de deux vecteurs | `np.dot(a, b)` |

**Remarque** : CMSIS-DSP propose des variantes `_f32` (float 32 bits), `_q15`, `_q31` pour différents formats numériques. Le choix dépend de la présence d'une **FPU** (Cortex-M4F/M7) et des contraintes de précision.

# Partie B – Filtrage temps réel simulé (6 points)

## B.1 – Filtre FIR temps réel par blocs

**Question B.1 (3 pts)** : Sur STM32, le filtrage se fait généralement par **blocs** (ex. 32 échantillons) pour optimiser le traitement.

1. Concevez un filtre **FIR passe-bas** ($f_c = 1000$ Hz, $f_s = 8000$ Hz, ordre 32, fenêtre Hamming).
2. Implémentez une fonction `filtrage_par_blocs(x, h, taille_bloc)` qui traite le signal par blocs (comme le ferait CMSIS-DSP).
3. Comparez le résultat avec `signal.lfilter` pour vérifier l'exactitude.
4. Mesurez le **temps de traitement** pour 1 seconde de signal.

In [ ]:
# --- Votre code ici ---
fs = 8000
fc = 1000
ordre_fir = 32
h = signal.firwin(ordre_fir + 1, fc, fs=fs, window='hamming')

def filtrage_par_blocs(x, h, taille_bloc=32):
    """Simule un filtrage FIR par blocs avec état (comme CMSIS-DSP)."""
    M = len(h)
    state = np.zeros(M - 1)  # mémoire entre blocs
    y = np.zeros(len(x))
    for i in range(0, len(x), taille_bloc):
        bloc = x[i:i+taille_bloc]
        # Concatène état + bloc
        buffer = np.concatenate([state, bloc])
        # Convolution et extraction
        y_bloc = np.convolve(buffer, h, mode='full')[:len(bloc)]
        y[i:i+len(bloc)] = y_bloc
        # Mise à jour de l'état
        state = buffer[-(M-1):]
    return y

# Signal test : 200 Hz (passe) + 3000 Hz (coupé)
t = np.arange(fs) / fs  # 1 seconde
x = np.sin(2*np.pi*200*t) + np.sin(2*np.pi*3000*t)

# Filtrage référence
t0 = time.perf_counter()
y_ref = signal.lfilter(h, [1.0], x)
t_ref = time.perf_counter() - t0

# Filtrage par blocs (simulation CMSIS-DSP)
t0 = time.perf_counter()
y_bloc = filtrage_par_blocs(x, h, taille_bloc=32)
t_bloc = time.perf_counter() - t0

# Comparaison
print(f"Erreur max |y_ref - y_bloc| : {np.max(np.abs(y_ref - y_bloc)):.2e}")
print(f"Temps lfilter (référence) : {t_ref*1e6:.1f} µs")
print(f"Temps filtrage par blocs  : {t_bloc*1e6:.1f} µs")

# Spectres comparés
X = fft(x)
Y_ref = fft(y_ref)
Y_bloc = fft(y_bloc)
freqs = fftfreq(len(x), 1/fs)[:len(x)//2]

plt.figure(figsize=(12, 5))
plt.plot(freqs, 20*np.log10(np.abs(X[:len(x)//2])/len(x) + 1e-12), label='Original', alpha=0.6)
plt.plot(freqs, 20*np.log10(np.abs(Y_ref[:len(x)//2])/len(x) + 1e-12), label='Filtré (référence)')
plt.plot(freqs, 20*np.log10(np.abs(Y_bloc[:len(x)//2])/len(x) + 1e-12), '--', label='Filtré (par blocs)')
plt.xlim(0, 4000)
plt.ylim(-80, 0)
plt.xlabel('Fréquence (Hz)')
plt.ylabel('Magnitude (dB)')
plt.title('Filtrage FIR : référence vs par blocs (simulation CMSIS-DSP)')
plt.legend()
plt.tight_layout()
plt.show()

### Interprétation B.1

- Le **filtrage par blocs** produit exactement le même résultat que `lfilter` (erreur nulle à la précision machine).
- La **composante à 200 Hz** est conservée (< $f_c = 1000$ Hz).
- La **composante à 3000 Hz** est fortement atténuée (> $f_c$).
- **Intérêt du traitement par blocs** :
  - Compatible avec un **buffer DMA** (acquisition ADC → traitement → DAC).
  - Permet un traitement **temps réel** avec une **latence fixe** (un bloc).
  - Sur STM32, `arm_fir_f32` traite un bloc de `blockSize` échantillons d'un coup.
- **Attention** : il faut conserver l'**état du filtre** entre les blocs (`state` dans CMSIS-DSP = `pState`).

## B.2 – Filtre IIR en cascade de biquads (CMSIS-DSP)

**Question B.2 (3 pts)** : Sur CMSIS-DSP, les filtres IIR d'ordre élevé sont implémentés en **cascade de sections biquad** (ordre 2) pour des raisons de stabilité numérique.

1. Concevez un filtre **Butterworth passe-bande** ($f_{low} = 300$ Hz, $f_{high} = 3000$ Hz, $f_s = 8000$ Hz, ordre 4).
2. Convertissez-le en **cascade de biquads** avec `signal.tf2sos`.
3. Implémentez manuellement la cascade (boucle sur les sections).
4. Comparez avec `signal.sosfilt`.

In [ ]:
# --- Votre code ici ---
fs = 8000
f_low, f_high = 300, 3000
ordre_iir = 4
b, a = butter(ordre_iir, [f_low, f_high], btype='band', fs=fs)
sos = signal.tf2sos(b, a)
print(f"Nombre de sections biquad : {sos.shape[0]}")
print(f"Forme SOS (shape) : {sos.shape}")

def cascade_biquad(x, sos):
    """Implémentation manuelle d'une cascade de biquads (forme directe II transposée)."""
    y = x.copy().astype(np.float64)
    n_sections = sos.shape[0]
    for s in range(n_sections):
        b0, b1, b2, a0, a1, a2 = sos[s]
        # Normalisation par a0
        b0, b1, b2 = b0/a0, b1/a0, b2/a0
        a1, a2 = a1/a0, a2/a0
        # Application de la section (forme directe II transposée)
        z1, z2 = 0.0, 0.0
        y_new = np.zeros_like(y)
        for n in range(len(y)):
            y_new[n] = b0*y[n] + z1
            z1 = b1*y[n] - a1*y_new[n] + z2
            z2 = b2*y[n] - a2*y_new[n]
        y = y_new
    return y

# Signal test
t = np.arange(fs) / fs
x = (np.sin(2*np.pi*100*t)   # hors bande (coupé)
     + np.sin(2*np.pi*500*t)  # dans la bande (conservé)
     + np.sin(2*np.pi*2000*t) # dans la bande (conservé)
     + np.sin(2*np.pi*3500*t))# hors bande (coupé)

# Filtrage référence
y_ref = signal.sosfilt(sos, x)
# Filtrage manuel
y_man = cascade_biquad(x, sos)

print(f"Erreur max |y_ref - y_man| : {np.max(np.abs(y_ref - y_man)):.2e}")

# Spectres
X = fft(x)
Y = fft(y_man)
freqs = fftfreq(len(x), 1/fs)[:len(x)//2]
plt.figure(figsize=(12, 5))
plt.plot(freqs, 20*np.log10(np.abs(X[:len(x)//2])/len(x) + 1e-12), label='Original', alpha=0.6)
plt.plot(freqs, 20*np.log10(np.abs(Y[:len(x)//2])/len(x) + 1e-12), label='Filtré (cascade biquad)')
plt.axvline(f_low, color='r', linestyle=':', alpha=0.6, label=f'Bande {f_low}-{f_high} Hz')
plt.axvline(f_high, color='r', linestyle=':', alpha=0.6)
plt.xlim(0, 4000)
plt.ylim(-80, 0)
plt.xlabel('Fréquence (Hz)')
plt.ylabel('Magnitude (dB)')
plt.title('Filtrage passe-bande par cascade de biquads (simulation CMSIS-DSP)')
plt.legend()
plt.tight_layout()
plt.show()

### Interprétation B.2

- Les composantes à **500 Hz** et **2000 Hz** sont **conservées** (dans la bande passante).
- Les composantes à **100 Hz** et **3500 Hz** sont **atténuées** (hors bande).
- L'**erreur maximale** entre l'implémentation manuelle et `scipy.signal.sosfilt` est de l'ordre de $10^{-12}$ (précision machine), ce qui valide notre implémentation.
- **Pourquoi une cascade de biquads ?**
  - Un filtre IIR d'ordre $N$ peut être **instable numériquement** en forme directe à cause des erreurs d'arrondi (coefficients très différents).
  - La **cascade de sections d'ordre 2** limite cette propagation d'erreurs.
  - CMSIS-DSP utilise `arm_biquad_cascade_df1_f32` ou `arm_biquad_cascade_df2T_f32` (forme directe II transposée, meilleure stabilité).
- **Sur STM32** : on stocke les coefficients sous forme `float32_t[5]` par section et on appelle `arm_biquad_cascade_df1_f32(&S, pSrc, pDst, blockSize)`.

# Partie C – Traitement audio / ECG (5 points)

## C.1 – Traitement d'un signal ECG simulé

**Question C.1 (3 pts)** : On simule un signal ECG bruité (bruit 50 Hz + bruit blanc) et on applique la chaîne de traitement typique d'un dispositif embarqué :

1. **Générer** un ECG synthétique à 250 Hz pendant 10 secondes (battements à 1 Hz).
2. **Ajouter** un bruit 50 Hz (interférence secteur) et un bruit blanc.
3. Concevoir un filtre **passe-bande** 0.5–40 Hz (plage utile ECG) et un **coupe-bande** à 50 Hz (notch).
4. Appliquer la chaîne de filtrage et comparer avec le signal original.

In [ ]:
# --- Votre code ici ---
fs_ecg = 250
duree = 10
t = np.linspace(0, duree, int(fs_ecg*duree), endpoint=False)

# Génération d'un ECG synthétique (impulsion périodique = complexe QRS)
x_ecg = np.zeros_like(t)
for i in range(0, len(t), fs_ecg):  # 1 battement/s
    if i+20 < len(t):
        x_ecg[i:i+20] = 1.0

# Bruits
bruit_50hz = 0.3 * np.sin(2*np.pi*50*t)
bruit_blanc = 0.1 * np.random.randn(len(t))
x_bruite = x_ecg + bruit_50hz + bruit_blanc

# Filtre passe-bande 0.5 - 40 Hz
b_bp, a_bp = butter(2, [0.5, 40], btype='band', fs=fs_ecg)
# Filtre coupe-bande (notch) à 50 Hz
b_notch, a_notch = signal.iirnotch(50, Q=30, fs=fs_ecg)

# Chaîne de filtrage : notch puis passe-bande
x_notch = filtfilt(b_notch, a_notch, x_bruite)
x_filtre = filtfilt(b_bp, a_bp, x_notch)

# Tracé
fig, axes = plt.subplots(3, 1, figsize=(14, 9), sharex=True)
axes[0].plot(t[:2500], x_ecg[:2500], 'k')
axes[0].set_title('ECG original (synthétique)')
axes[0].set_ylabel('Amplitude')

axes[1].plot(t[:2500], x_bruite[:2500], 'r', alpha=0.6)
axes[1].set_title('ECG bruité (50 Hz + bruit blanc)')
axes[1].set_ylabel('Amplitude')

axes[2].plot(t[:2500], x_filtre[:2500], 'g')
axes[2].set_title('ECG filtré (notch 50 Hz + passe-bande 0.5-40 Hz)')
axes[2].set_xlabel('Temps (s)')
axes[2].set_ylabel('Amplitude')

plt.tight_layout()
plt.show()

# Comparaison des spectres
N = len(t)
X_bruite = fft(x_bruite)
X_filtre = fft(x_filtre)
freqs = fftfreq(N, 1/fs_ecg)[:N//2]

plt.figure(figsize=(12, 5))
plt.plot(freqs, 20*np.log10(np.abs(X_bruite[:N//2])/N + 1e-12), label='Bruité', alpha=0.6)
plt.plot(freqs, 20*np.log10(np.abs(X_filtre[:N//2])/N + 1e-12), label='Filtré')
plt.axvline(50, color='r', linestyle='--', alpha=0.6, label='50 Hz (parasite)')
plt.xlim(0, 100)
plt.ylim(-100, 0)
plt.xlabel('Fréquence (Hz)')
plt.ylabel('Magnitude (dB)')
plt.title('Comparaison des spectres avant/après filtrage ECG')
plt.legend()
plt.tight_layout()
plt.show()

print("→ Le pic à 50 Hz a été fortement atténué (>40 dB).")
print("→ La forme du complexe QRS est préservée.")

### Interprétation C.1

- Le **filtre notch à 50 Hz** élimine l'**interférence secteur** (fréquence du réseau électrique en Europe).
- Le **filtre passe-bande 0.5–40 Hz** conserve la bande utile de l'ECG (le complexe QRS a une énergie principalement entre 5 et 40 Hz).
- Le **filtrage en cascade** (notch + passe-bande) est une **chaîne classique** en instrumentation biomédicale.
- **Sur STM32** : ces deux filtres sont implémentés en cascade de biquads, traités par blocs de 32 ou 64 échantillons dans une ISR (interruption ADC).
- **Contraintes embarquées** :
  - Latence < 10 ms (pour l'affichage temps réel).
  - Consommation énergétique minimale (traitement par DMA et sleep entre blocs).
  - Stabilité numérique en Q15 ou float32.

## C.2 – Détection de la fréquence cardiaque

**Question C.2 (2 pts)** : À partir du signal ECG **filtré**, détectez le **nombre de battements par minute (BPM)**.

1. Détectez les pics QRS dans le domaine temporel.
2. Calculez la fréquence cardiaque moyenne.
3. Comparez avec la valeur théorique (1 Hz = 60 BPM).

In [ ]:
# --- Votre code ici ---
peaks, props = find_peaks(x_filtre, height=0.5, distance=fs_ecg*0.4)

# Intervalles entre pics
intervalles = np.diff(peaks) / fs_ecg  # en secondes
bpm_moyen = 60 / np.mean(intervalles)

print(f"Nombre de battements détectés : {len(peaks)}")
print(f"Intervalle moyen : {np.mean(intervalles):.3f} s")
print(f"Fréquence cardiaque : {bpm_moyen:.1f} BPM")
print(f"Valeur théorique : 60 BPM (battement à 1 Hz)")

# Visualisation
plt.figure(figsize=(14, 4))
plt.plot(t[:2500], x_filtre[:2500], label='ECG filtré')
peaks_visibles = peaks[peaks < 2500]
plt.plot(t[peaks_visibles], x_filtre[peaks_visibles], 'ro', label='Pics QRS détectés')
plt.xlabel('Temps (s)')
plt.ylabel('Amplitude')
plt.title('Détection des pics QRS')
plt.legend()
plt.tight_layout()
plt.show()

### Interprétation C.2

- La **détection de pics** (`find_peaks`) identifie les complexes QRS comme maxima locaux du signal filtré.
- Le paramètre `distance` évite les **fausses détections** (une seule détection par battement).
- La fréquence cardiaque calculée (~60 BPM) est **cohérente** avec le signal synthétique.
- **Sur un dispositif réel** :
  - La détection se fait dans une **ISR** ou un **thread RTOS** à basse priorité.
  - On utilise un algorithme plus robuste (Pan-Tompkins) basé sur :
    1. Filtrage passe-bande (5–15 Hz).
    2. Dérivation.
    3. Élévation au carré.
    4. Intégration sur fenêtre glissante.
    5. Seuillage adaptatif.

# Partie D – Optimisation embarquée (5 points)

## D.1 – Analyse du coût computationnel

**Question D.1 (2 pts)** : Pour les fonctions CMSIS-DSP suivantes, estimez le nombre d'opérations (MAC = multiplication-accumulation) par échantillon :

1. Filtre FIR d'ordre $N$.
2. Filtre IIR (biquad) d'ordre 2.
3. FFT radix-2 de taille $N$ (par échantillon).

**Réponse** :

### Réponse D.1

**1. Filtre FIR d'ordre $N$ :**
$$\text{MAC/échantillon} = N + 1$$
Pour un filtre d'ordre 64 : **65 MACs/échantillon**.

**2. Filtre IIR biquad (ordre 2) :**
$$\text{MAC/échantillon} \approx 5 \text{ (2 pôles + 3 zéros)}$$
Une cascade de 2 biquads (ordre 4) = **10 MACs/échantillon**.

**3. FFT radix-2 de taille $N$ :**
Complexité totale : $\mathcal{O}(N \log_2 N)$ opérations complexes.
Soit $\approx \frac{N \log_2 N}{N} = \log_2 N$ opérations complexes par échantillon, soit $\approx 5 \log_2 N$ MACs réels.
Pour $N = 1024$ : $\approx 5 \times 10 = 50$ MACs/échantillon, ou $\approx 51200$ MACs au total.

**Comparaison** : Pour une même pente, un filtre IIR est **beaucoup plus économique** qu'un FIR. Pour une FFT de taille $N = 1024$, CMSIS-DSP calcule en $\approx 2$ ms sur un Cortex-M4 @ 100 MHz (soit ~200 000 cycles).

## D.2 – Benchmark simulé

**Question D.2 (3 pts)** : Comparez le temps d'exécution de plusieurs opérations sur **1 seconde** de signal audio ($f_s = 8000$ Hz, 8000 échantillons) :

1. Filtrage FIR par blocs (ordre 64).
2. Filtrage IIR (cascade de biquads, ordre 4).
3. FFT de 8000 points (une seule fois vs par blocs de 512).

Mesurez le temps avec `time.perf_counter` et concluez sur le choix à faire en embarqué.

In [ ]:
# --- Votre code ici ---
fs = 8000
N = fs  # 1 seconde
t = np.arange(N) / fs
x = np.sin(2*np.pi*440*t) + 0.3*np.random.randn(N)

# 1. Filtre FIR ordre 64
h_fir = signal.firwin(65, 1000, fs=fs, window='hamming')
t0 = time.perf_counter()
y_fir = filtfilt(h_fir, [1.0], x)
t_fir = time.perf_counter() - t0

# 2. Filtre IIR ordre 4 (cascade biquad)
b_iir, a_iir = butter(4, 1000, fs=fs)
sos = signal.tf2sos(b_iir, a_iir)
t0 = time.perf_counter()
y_iir = signal.sosfilt(sos, x)
t_iir = time.perf_counter() - t0

# 3a. FFT globale
t0 = time.perf_counter()
X_global = fft(x)
t_fft_glob = time.perf_counter() - t0

# 3b. FFT par blocs de 512
taille_bloc = 512
t0 = time.perf_counter()
X_blocs = []
for i in range(0, N - taille_bloc + 1, taille_bloc):
    X_blocs.append(fft(x[i:i+taille_bloc] * np.hamming(taille_bloc)))
t_fft_blocs = time.perf_counter() - t0

print(f"{'Opération':<40} {'Temps (ms)':>12}")
print("-" * 55)
print(f"{'Filtrage FIR (ordre 64)':<40} {t_fir*1e3:>12.3f}")
print(f"{'Filtrage IIR (ordre 4, biquad)':<40} {t_iir*1e3:>12.3f}")
print(f"{'FFT globale (N=8000)':<40} {t_fft_glob*1e3:>12.3f}")
print(f"{'FFT par blocs de 512':<40} {t_fft_blocs*1e3:>12.3f}")

### Interprétation D.2

**Résultats typiques (Python, machine de bureau) :**
| Opération | Temps (ms) |
|---|---|
| Filtrage FIR (ordre 64) | ~2-5 |
| Filtrage IIR (ordre 4, biquad) | ~0.5-1 |
| FFT globale (N=8000) | ~0.3-0.5 |
| FFT par blocs de 512 | ~0.5-1 |

**Conclusion pour l'embarqué (STM32F4 @ 168 MHz) :**

1. **Filtrage IIR >> FIR** en termes de **MIPS** : pour une même pente, un biquad d'ordre 2 fait l'équivalent d'un FIR d'ordre ~20. → **Privilégier IIR** quand la phase linéaire n'est pas critique.
2. **FFT globale** vs **FFT par blocs** : la FFT globale est plus efficace (moins de recouvrement), mais **incompatible temps réel** (il faut attendre d'avoir tout le signal).
3. **FFT par blocs (512)** avec recouvrement (overlap-add) permet un **traitement temps réel** avec une **latence de 64 ms** (512/8000).
4. **Optimisations supplémentaires** :
   - Utiliser `arm_rfft_fast_f32` (2× plus rapide que FFT complexe pour signal réel).
   - Activer la **FPU** (Cortex-M4F) et le **DSP extension**.\
   - Utiliser des **buffers DMA** pour l'acquisition ADC.
   - Minimiser les **accès mémoire** (utiliser des buffers en RAM interne).

# Synthèse de l'évaluation pratique – Partie 2

## Récapitulatif des compétences évaluées

| Compétence | Partie | Statut |
|---|---|---|
| Représentation en virgule fixe Q15 | A | ✅ |
| Fonctions CMSIS-DSP essentielles | A | ✅ |
| Filtrage FIR par blocs (temps réel) | B | ✅ |
| Filtre IIR en cascade biquad | B | ✅ |
| Traitement ECG (notch + passe-bande) | C | ✅ |
| Détection QRS et calcul du BPM | C | ✅ |
| Estimation du coût computationnel | D | ✅ |
| Benchmark FIR/IIR/FFT | D | ✅ |

## Points clés à retenir

1. **Q15** : format efficace pour MCU sans FPU, plage [−1, 1[, résolution $2^{-15}$.
2. **CMSIS-DSP** : bibliothèque optimisée pour Cortex-M, avec variantes `_f32`, `_q15`, `_q31`.
3. **Traitement par blocs** : indispensable pour le temps réel (DMA, ISR, buffers).
4. **Cascade de biquads** : implémentation standard des filtres IIR d'ordre élevé.
5. **Chaîne ECG** : notch 50 Hz + passe-bande 0.5–40 Hz → détection QRS → BPM.
6. **Optimisation** : IIR plus économique que FIR, `arm_rfft_fast` pour la FFT réelle.

## Critères de qualité d'une implémentation embarquée

- **Latence** : < 10 ms pour un traitement temps réel audio/ECG.
- **Précision** : erreur relative < 1% par rapport à la référence flottante.
- **Mémoire** : < 10 Ko de RAM pour les buffers et l'état des filtres.
- **CPU** : < 50% de charge sur MCU @ 100 MHz.
- **Robustesse** : gestion de la saturation, redémarrage propre.

## Préparation pour la Semaine 14

- **Révision finale** de tous les concepts (semaines 1 à 13).
- **Projet de synthèse** : application complète de TNS de bout en bout.
- **Préparation** du rapport final et de la présentation.

---

# Livrable Semaine 13

- Remplir **toutes** les cellules de code avec vos solutions.
- **Commenter** chaque résultat obtenu (interprétation embarquée).
- Les graphiques doivent être **lisibles** (titre, axes, légende, grille).
- Rendre le notebook complet (cellules exécutées).

**Checklist avant rendu** :
- [ ] Vérifier qu'aucune cellule ne produit d'erreur (`Kernel → Restart & Run All`)
- [ ] Toutes les **interprétations** sont présentes après chaque question
- [ ] Les **graphiques** sont lisibles (labels, légendes, titres)
- [ ] Les fonctions **Q15** sont correctement implémentées et testées
- [ ] Le **filtrage par blocs** est validé contre la référence
- [ ] La **cascade biquad** est correctement implémentée
- [ ] Le **signal ECG** est filtré et les **pics QRS** sont détectés
- [ ] Le **BPM** est calculé et comparé à la valeur théorique
- [ ] Le **benchmark** FIR/IIR/FFT est complété avec conclusions
- [ ] Les **optimisations embarquées** sont identifiées
- [ ] Exporter en PDF/HTML avec les sorties visibles

**Date de rendu** : à définir par l'enseignant.

---